# LHIPA on AdSERP: Pupillometric Cognitive Load During Search

Implements the **Low/High Index of Pupillary Activity** (Duchowski et al., CHI 2020) on the AdSERP dataset (Latifzadeh, Gwizdka & Leiva, SIGIR 2025). LHIPA uses discrete wavelet decomposition to isolate cognitive-load-related pupil diameter changes from luminance-related ones — critical for naturalistic web browsing where page luminance varies constantly.

**What this notebook provides:**
1. A clean, documented LHIPA implementation for Gazepoint GP3 HD data (150 Hz)
2. Validation against behavioral proxies (trial duration, regression count, click position)
3. Per-result LHIPA extending the algorithm to finer temporal grain than prior work
4. Trial-level and per-result cognitive load signatures during SERP browsing

**Prior work:** Shi, Jayawardena & Gwizdka (CHIIR 2025) applied LHIPA to document relevance assessment using a Tobii TX-300 (300 Hz) and experimentally varied relevance. We apply the same algorithm to naturalistic search behavior at 150 Hz, testing whether LHIPA tracks the cognitive demands of the search process itself — orientation, evaluation, regression, and click commitment.

**Algorithm reference:** Duchowski, A. T., Krejtz, K., Zuber, S., & Krejtz, I. (2020). The Low/High Index of Pupillary Activity. CHI '20. [doi:10.1145/3313831.3376394](https://doi.org/10.1145/3313831.3376394)

**Dataset:** AdSERP pupil data (`pupil-data/*.csv`): timestamp, BPOGX, BPOGY, LPD, LPV, RPD, RPV at 150 Hz from Gazepoint GP3 HD.

In [ ]:
# Shared data loading — see data_loader.py for all utilities
from data_loader import *
setup_plotting()
import os, csv, json, time
from collections import defaultdict
import numpy as np
import pywt
import matplotlib.pyplot as plt
from scipy import stats


## LHIPA Algorithm

The algorithm (Duchowski et al., 2020) isolates cognitive pupil responses from luminance-driven responses using wavelet decomposition:

1. **Preprocessing**: Remove blinks (LPV/RPV=0) with a 200ms exclusion window. Average valid eyes. Interpolate over gaps.
2. **Wavelet decomposition**: Symlet-16 (`sym16`) with periodization mode. Extract detail coefficients at level 1 (high-frequency, hif) and level max_level//2 (low-frequency, lof).
3. **Normalize**: Scale each band by `1/sqrt(2^j)` where j is the decomposition level.
4. **Ratio**: Compute element-wise `|cD_L| / mean(|cD_H|)` aligning subsampling rates.
5. **Modulus maxima**: Find local peaks of the absolute ratio signal.
6. **Threshold**: Hard threshold at `sigma * sqrt(2 * log2(N))` where sigma is MAD-based.
7. **LHIPA**: Count of surviving maxima divided by signal length.

**Interpretation**: Higher LHIPA = lower cognitive load. The index measures how much high-frequency pupil activity (cognitive) dominates over low-frequency activity (luminance). When cognitive load is high, pupil fluctuations are suppressed, producing fewer surviving modulus maxima.

**150 Hz note**: Duchowski et al. validated on 300 Hz Tobii data. The Gazepoint GP3 HD at 150 Hz provides half the temporal resolution, shifting the maximum decomposition level down by ~1. The sym16 wavelet (32 coefficients) spans ~213ms at 150 Hz vs ~107ms at 300 Hz. Both are within the range of cognitive pupil responses (200-2000ms).

### Key Measures

| Measure | Definition | Units | Interpretation |
|---------|-----------|-------|----------------|
| LHIPA | Low/High Index of Pupillary Activity — ratio of low-frequency to high-frequency power in pupil diameter signal (Symlet-16 wavelet decomposition) | ratio | Lower = higher cognitive load; isolates cognitive demand from luminance changes |
| Trial-level LHIPA | Mean LHIPA across the full trial duration | ratio | Correlates with trial duration (rho=-0.65), regression count (rho=-0.44) |
| Per-position LHIPA | LHIPA computed during fixations on results at each rank position | ratio | Flat across click positions 0–8 (delta = 0.0008); step down at boundary 9–10. Aggregate ρ = -0.90 on N=10 position means is driven by boundary step. Trial-level ρ = -0.088 (N ≈ 2,700). See CHANGELOG v9. |


In [ ]:
# ── LHIPA Implementation ──────────────────────────────────────────────

def remove_blinks(timestamps, pupil_diameters, validity, exclusion_ms=200):
    """Remove blink samples and interpolate gaps.
    
    Blinks: validity=0 or pupil_diameter<=0.
    Exclusion window: 200ms around each invalid sample.
    Returns (clean_timestamps, clean_pupil) or (None, None) if >50% invalid.
    """
    ts = np.array(timestamps, dtype=float)
    pd = np.array(pupil_diameters, dtype=float)
    val = np.array(validity, dtype=int)
    
    invalid = (val == 0) | (pd <= 0)
    
    if invalid.any():
        invalid_times = ts[invalid]
        for it in invalid_times:
            mask = (ts >= it - exclusion_ms) & (ts <= it + exclusion_ms)
            invalid[mask] = True
    
    if invalid.sum() > len(invalid) * 0.5:
        return None, None
    
    valid_mask = ~invalid
    if valid_mask.sum() < 50:
        return None, None
    
    clean_pd = pd.copy()
    valid_indices = np.where(valid_mask)[0]
    invalid_indices = np.where(invalid)[0]
    
    if len(invalid_indices) > 0 and len(valid_indices) > 1:
        clean_pd[invalid_indices] = np.interp(
            ts[invalid_indices], ts[valid_indices], pd[valid_indices]
        )
    
    return ts, clean_pd


def compute_lhipa(pupil_signal, wavelet='sym16'):
    """Compute LHIPA for a pupil diameter time series.
    
    Args:
        pupil_signal: 1D array of pupil diameter values (blink-cleaned).
        wavelet: Wavelet to use (default: sym16, per Duchowski et al.).
    
    Returns:
        LHIPA value (float). Higher = lower cognitive load.
        None if signal is too short for decomposition.
    """
    signal = np.array(pupil_signal, dtype=float)
    n = len(signal)
    
    if n < 64:
        return None
    
    w = pywt.Wavelet(wavelet)
    max_level = pywt.dwt_max_level(n, w.dec_len)
    
    if max_level < 2:
        return None
    
    hif = 1
    lof = max(max_level // 2, 2)
    
    if lof <= hif:
        lof = hif + 1
    if lof > max_level:
        return None
    
    # Detail coefficients at high and low frequency levels
    cD_H = pywt.downcoef('d', signal, wavelet, mode='per', level=hif)
    cD_L = pywt.downcoef('d', signal, wavelet, mode='per', level=lof)
    
    # Normalize by 1/sqrt(2^j)
    cD_H = cD_H / np.sqrt(2**hif)
    cD_L = cD_L / np.sqrt(2**lof)
    
    # Compute L/H ratio, aligning subsampling rates
    ratio_factor = max(len(cD_H) // len(cD_L), 1)
    ratio = np.zeros(len(cD_L))
    for i in range(len(cD_L)):
        start = i * ratio_factor
        end = min(start + ratio_factor, len(cD_H))
        h_mean = np.mean(np.abs(cD_H[start:end]))
        if h_mean > 1e-10:
            ratio[i] = np.abs(cD_L[i]) / h_mean
    
    if len(ratio) < 3:
        return None
    
    # Modulus maxima (local peaks of |ratio|)
    abs_ratio = np.abs(ratio)
    maxima = []
    for i in range(1, len(abs_ratio) - 1):
        if abs_ratio[i] > abs_ratio[i-1] and abs_ratio[i] > abs_ratio[i+1]:
            maxima.append(abs_ratio[i])
    
    if not maxima:
        return 0.0
    
    maxima = np.array(maxima)
    
    # Hard threshold: sigma * sqrt(2 * log2(N))
    sigma = np.median(np.abs(cD_H)) / 0.6745  # MAD-based noise estimate
    threshold = sigma * np.sqrt(2 * np.log2(max(n, 2)))
    
    n_surviving = int(np.sum(maxima > threshold))
    
    return n_surviving / n


def load_pupil_trial(trial_id):
    """Load and preprocess pupil data for one trial.
    
    Returns dict with clean signal, timestamps, gaze coordinates,
    and per-sample combined pupil diameter. None if unusable.
    """
    path = os.path.join(PUPIL_DIR, f'{trial_id}.csv')
    ts, bx, by, lpd, lpv, rpd, rpv = [], [], [], [], [], [], []
    
    with open(path) as f:
        for row in csv.DictReader(f):
            try:
                ts.append(int(float(row['timestamp'])))
                bx.append(float(row['BPOGX']))
                by.append(float(row['BPOGY']))
                lpd.append(float(row['LPD']))
                lpv.append(int(row['LPV']))
                rpd.append(float(row['RPD']))
                rpv.append(int(row['RPV']))
            except:
                continue
    
    if len(ts) < 100:
        return None
    
    # Combine eyes: mean of valid, single eye if only one valid
    mean_pd = []
    combined_val = []
    for i in range(len(ts)):
        lv, rv = lpv[i], rpv[i]
        if lv and rv:
            mean_pd.append((lpd[i] + rpd[i]) / 2)
            combined_val.append(1)
        elif lv:
            mean_pd.append(lpd[i])
            combined_val.append(1)
        elif rv:
            mean_pd.append(rpd[i])
            combined_val.append(1)
        else:
            mean_pd.append(0)
            combined_val.append(0)
    
    clean_ts, clean_pd = remove_blinks(ts, mean_pd, combined_val)
    if clean_ts is None:
        return None
    
    return {
        'timestamps': ts,
        'bpogx': bx,
        'bpogy': by,
        'clean_ts': clean_ts,
        'clean_pd': clean_pd,
        'raw_pd': mean_pd,
        'validity': combined_val,
    }

In [ ]:
# ── Compute trial-level LHIPA for all trials ─────────────────────────

print('Computing trial-level LHIPA...')
t0 = time.time()

trial_lhipa = {}  # tid → {lhipa, mean_pd, n_samples, n_valid_pct, duration_s}
failed = 0

for tid in trial_ids:
    pupil = load_pupil_trial(tid)
    if pupil is None:
        failed += 1
        continue
    
    lhipa = compute_lhipa(pupil['clean_pd'])
    if lhipa is None:
        failed += 1
        continue
    
    duration_s = (pupil['clean_ts'][-1] - pupil['clean_ts'][0]) / 1000
    valid_pct = np.mean(pupil['validity']) * 100
    
    trial_lhipa[tid] = {
        'lhipa': lhipa,
        'mean_pd': float(np.mean(pupil['clean_pd'])),
        'n_samples': len(pupil['clean_pd']),
        'valid_pct': valid_pct,
        'duration_s': duration_s,
    }

elapsed = time.time() - t0
lhipa_vals = [v['lhipa'] for v in trial_lhipa.values()]

print(f'Computed: {len(trial_lhipa)} trials in {elapsed:.1f}s ({failed} failed)')
print(f'LHIPA: mean={np.mean(lhipa_vals):.6f}, median={np.median(lhipa_vals):.6f}, '
      f'SD={np.std(lhipa_vals):.6f}, range=[{np.min(lhipa_vals):.6f}, {np.max(lhipa_vals):.6f}]')
print(f'Valid pupil %: mean={np.mean([v["valid_pct"] for v in trial_lhipa.values()]):.1f}%')

# Export for notebook 11 (individual differences)
lhipa_export = {tid: {'lhipa': v['lhipa']} for tid, v in trial_lhipa.items()}
with open('lhipa_per_trial.json', 'w') as f:
    json.dump(lhipa_export, f)
print(f'Exported lhipa_per_trial.json ({len(lhipa_export)} trials)')

In [ ]:
# ── LHIPA distribution ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('LHIPA Distribution (Trial-Level)', fontsize=14, fontweight='bold')

# (a) Histogram
ax = axes[0]
ax.hist(lhipa_vals, bins=50, color='#6366f1', alpha=0.7, edgecolor='white')
ax.axvline(np.mean(lhipa_vals), color='#dc2626', linestyle='--', label=f'Mean: {np.mean(lhipa_vals):.4f}')
ax.axvline(np.median(lhipa_vals), color='#f59e0b', linestyle='--', label=f'Median: {np.median(lhipa_vals):.4f}')
ax.set_xlabel('LHIPA')
ax.set_ylabel('Count')
ax.set_title('Distribution')
ax.legend(fontsize=8)

# (b) LHIPA vs trial duration
ax = axes[1]
durs = [v['duration_s'] for v in trial_lhipa.values()]
ax.scatter(durs, lhipa_vals, alpha=0.1, s=3, color='#6366f1')
r_dur, p_dur = stats.spearmanr(durs, lhipa_vals)
ax.set_xlabel('Trial duration (s)')
ax.set_ylabel('LHIPA')
ax.set_title(f'LHIPA vs duration (ρ={r_dur:.3f}, p={p_dur:.1e})')
ax.set_xlim(0, 60)

# (c) LHIPA vs signal quality
ax = axes[2]
vpcts = [v['valid_pct'] for v in trial_lhipa.values()]
ax.scatter(vpcts, lhipa_vals, alpha=0.1, s=3, color='#6366f1')
ax.set_xlabel('Valid pupil samples (%)')
ax.set_ylabel('LHIPA')
ax.set_title('Signal quality check')

plt.tight_layout()
plt.savefig('plot_lhipa1_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── Validation: LHIPA vs behavioral proxies ──────────────────────────
#
# If LHIPA measures cognitive load, it should correlate with behavioral
# measures known to reflect task difficulty:
#   - Trial duration (longer = harder)
#   - Scroll regression count (more = harder decision)
#   - Click position (deeper = harder foraging)
#   - Number of fixations (more evaluation = harder)

# Load behavioral data per trial
print('Loading behavioral data...')

behavioral = {}  # tid → {duration_s, n_regressions, click_pos, n_fixations}

for tid in trial_lhipa:
    mouse_path = os.path.join(MOUSE_DIR, f'{tid}.csv')
    if not os.path.exists(mouse_path):
        continue
    
    scrolls = []
    click_t = None
    click_y = None
    all_ts = []
    
    with open(mouse_path) as f:
        for row in csv.DictReader(f):
            t = int(float(row['timestamp']))
            all_ts.append(t)
            if row['event'] == 'scroll':
                scrolls.append((t, float(row['ypos'])))
            if row['event'] == 'click' and click_t is None:
                click_t = t
                click_y = float(row['ypos'])
    
    if not all_ts or click_t is None:
        continue
    
    duration = (max(all_ts) - min(all_ts)) / 1000
    
    # Regression count (same 200ms/10px thresholds)
    n_reg = 0
    if len(scrolls) >= 2:
        gesture_start_y = scrolls[0][1]
        for i in range(1, len(scrolls)):
            if scrolls[i][0] - scrolls[i-1][0] > 200:
                if scrolls[i-1][1] - gesture_start_y < -10:
                    n_reg += 1
                gesture_start_y = scrolls[i][1]
        if scrolls[-1][1] - gesture_start_y < -10:
            n_reg += 1
    
    # Click position
    try:
        tree = ET.parse(os.path.join(METADATA_DIR, f'{tid}.xml'))
        doc_h = int(tree.find('.//document').text.split('x')[1])
        scr_h = int(tree.find('.//screen').text.split('x')[1])
    except:
        continue
    
    so = 0.0
    if scrolls:
        sts = [s[0] for s in scrolls]
        sys_ = [s[1] for s in scrolls]
        if click_t <= sts[0]: so = sys_[0]
        elif click_t >= sts[-1]: so = sys_[-1]
        else:
            idx = bisect_right(sts, click_t) - 1
            so = sys_[idx]
    
    header = 200
    per_res = (doc_h - 400) / 10
    tops = [header + i * per_res for i in range(10)]
    page_y = min(click_y, scr_h) + so
    click_pos = bisect_right(tops, page_y) - 1
    click_pos = max(0, min(click_pos, 9))
    
    # Fixation count
    n_fix = 0
    fix_path = os.path.join(FIXATION_DIR, f'{tid}.csv')
    if os.path.exists(fix_path):
        with open(fix_path) as f:
            n_fix = sum(1 for _ in csv.DictReader(f))
    
    behavioral[tid] = {
        'duration_s': duration,
        'n_regressions': n_reg,
        'click_pos': click_pos,
        'n_fixations': n_fix,
    }

# Compute correlations
shared = [tid for tid in trial_lhipa if tid in behavioral]
lh = np.array([trial_lhipa[tid]['lhipa'] for tid in shared])

print(f'\nTrials with LHIPA + behavioral data: {len(shared)}')
print(f'\n{"Behavioral measure":<25} {"Spearman ρ":>12} {"p":>12} {"Direction":>20}')

for name, key in [('Trial duration (s)', 'duration_s'), 
                   ('Regression count', 'n_regressions'),
                   ('Click position', 'click_pos'),
                   ('Fixation count', 'n_fixations')]:
    vals = np.array([behavioral[tid][key] for tid in shared])
    rho, p = stats.spearmanr(lh, vals)
    # Lower LHIPA = more load; higher behavioral = harder task
    # So negative rho = LHIPA validates (harder task → lower LHIPA)
    direction = 'VALIDATES' if rho < 0 else 'contra'
    sig = '**' if p < 0.001 else ('*' if p < 0.05 else '')
    print(f'{name:<25} {rho:>12.4f} {p:>12.2e} {direction:>20} {sig}')

In [ ]:
# ── Validation plots ──────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LHIPA Validation Against Behavioral Proxies', fontsize=14, fontweight='bold')

measures = [
    ('duration_s', 'Trial Duration (s)', (0, 60)),
    ('n_regressions', 'Scroll Regressions', (0, 12)),
    ('click_pos', 'Click Position', (-0.5, 9.5)),
    ('n_fixations', 'Fixation Count', (0, 200)),
]

for ax, (key, label, xlim) in zip(axes.flat, measures):
    vals = np.array([behavioral[tid][key] for tid in shared])
    rho, p = stats.spearmanr(lh, vals)
    
    ax.scatter(vals, lh, alpha=0.08, s=3, color='#6366f1')
    
    # Binned means
    if key == 'click_pos':
        for pos in range(10):
            mask = vals == pos
            if mask.sum() > 5:
                mean_lh = np.mean(lh[mask])
                se = np.std(lh[mask]) / np.sqrt(mask.sum())
                ax.errorbar(pos, mean_lh, yerr=1.96*se, fmt='o', color='#dc2626',
                           markersize=6, capsize=3, zorder=5)
    else:
        n_bins = 15
        bin_edges = np.linspace(xlim[0], xlim[1], n_bins + 1)
        for i in range(n_bins):
            mask = (vals >= bin_edges[i]) & (vals < bin_edges[i+1])
            if mask.sum() > 10:
                bc = (bin_edges[i] + bin_edges[i+1]) / 2
                ax.plot(bc, np.mean(lh[mask]), 'o', color='#dc2626', markersize=4, zorder=5)
    
    ax.set_xlabel(label)
    ax.set_ylabel('LHIPA (↑ = less load)')
    ax.set_title(f'ρ = {rho:.3f} (p = {p:.1e})')
    ax.set_xlim(xlim)

plt.tight_layout()
plt.savefig('plot_lhipa2_validation.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── LHIPA by click position (the strongest finding) ──────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LHIPA by Click Position (Flat 0–8, Boundary Step at 9–10)', fontsize=14, fontweight='bold')

# (a) Mean LHIPA by click position
ax = axes[0]
pos_means = []
pos_sems = []
pos_ns = []
for p in range(10):
    tids = [tid for tid in shared if behavioral[tid]['click_pos'] == p]
    if tids:
        vals = [trial_lhipa[tid]['lhipa'] for tid in tids]
        pos_means.append(np.mean(vals))
        pos_sems.append(np.std(vals) / np.sqrt(len(vals)))
        pos_ns.append(len(vals))
    else:
        pos_means.append(np.nan)
        pos_sems.append(0)
        pos_ns.append(0)

bars = ax.bar(range(10), pos_means, yerr=[1.96*s for s in pos_sems],
              color='#6366f1', alpha=0.7, capsize=4, edgecolor='white')
ax.set_xlabel('Clicked Result Position')
ax.set_ylabel('Mean LHIPA (↑ = less load)')
ax.set_title('LHIPA by Click Position')
ax.set_xticks(range(10))
for i, (bar, n) in enumerate(zip(bars, pos_ns)):
    if n > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'n={n}', ha='center', fontsize=7, color='gray')

# Spearman on means
valid_pos = [(p, m) for p, m in enumerate(pos_means) if not np.isnan(m)]
rho_cp, p_cp = stats.spearmanr([p for p, _ in valid_pos], [m for _, m in valid_pos])
ax.text(0.95, 0.95, f'ρ = {rho_cp:.3f}\np = {p_cp:.3e}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# (b) Per-participant LHIPA profile
ax = axes[1]
by_participant = defaultdict(list)
for tid in shared:
    pid = tid.split('-')[0]
    by_participant[pid].append(trial_lhipa[tid]['lhipa'])

participant_means = sorted([(pid, np.mean(vals)) for pid, vals in by_participant.items()],
                           key=lambda x: x[1])
ax.bar(range(len(participant_means)), [m for _, m in participant_means],
       color='#6366f1', alpha=0.7, edgecolor='white')
ax.set_xlabel('Participant (sorted by mean LHIPA)')
ax.set_ylabel('Mean LHIPA')
ax.set_title(f'Individual Differences (N={len(participant_means)} participants)')
ax.set_xticks([])

plt.tight_layout()
plt.savefig('plot_lhipa3_click_position.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'LHIPA × click position: ρ = {rho_cp:.4f} (p = {p_cp:.4e})')
print(f'Individual LHIPA range: {participant_means[0][1]:.4f} to {participant_means[-1][1]:.4f}')

> *Functions `result_bands` moved to `data_loader.py`.*

> **Minimum window caveat (Duchowski 2026).** The wavelet decomposition in LHIPA requires a minimum of 7.5–10 seconds for stable frequency band separation (Duchowski, PACM CGIT 2026, §5.2). Per-result segments in AdSERP are ~2 seconds (~300 samples at 150 Hz) — well below this threshold. The per-result LHIPA values below should be interpreted cautiously. For per-position cognitive load analysis with proper short-window support, see **notebook 14** which uses Duchowski's recommended Butterworth IIR method (1-second minimum window). The Butterworth method reverses the per-position pattern: cognitive load *decreases* with position, not increases — a finding only visible with a method validated for short windows.

In [ ]:
# ── Per-result LHIPA by position ──────────────────────────────────────

has_lhipa = [r for r in result_rows if r['result_lhipa'] is not None]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Per-Result LHIPA During SERP Browsing', fontsize=14, fontweight='bold')

# (a) LHIPA by position
ax = axes[0]
pos_means_r = []
pos_sems_r = []
for p in range(10):
    sub = [r['result_lhipa'] for r in has_lhipa if r['position'] == p]
    if sub:
        pos_means_r.append(np.mean(sub))
        pos_sems_r.append(np.std(sub) / np.sqrt(len(sub)))
    else:
        pos_means_r.append(np.nan)
        pos_sems_r.append(0)

ax.bar(range(10), pos_means_r, yerr=[1.96*s for s in pos_sems_r],
       color='#6366f1', alpha=0.7, capsize=4)
ax.set_xlabel('Result Position')
ax.set_ylabel('Mean LHIPA (↑ = less load)')
ax.set_title('LHIPA by Position')
ax.set_xticks(range(10))

# (b) Clicked vs non-clicked
ax = axes[1]
for p in range(10):
    c = [r['result_lhipa'] for r in has_lhipa if r['position'] == p and r['is_clicked'] == 1]
    nc = [r['result_lhipa'] for r in has_lhipa if r['position'] == p and r['is_clicked'] == 0]
    if c and nc:
        ax.plot(p, np.mean(c), 'o', color='#dc2626', markersize=6, zorder=5)
        ax.plot(p, np.mean(nc), 's', color='#2563eb', markersize=6, zorder=5)

ax.plot([], [], 'o', color='#dc2626', label='Clicked')
ax.plot([], [], 's', color='#2563eb', label='Non-clicked')
ax.set_xlabel('Result Position')
ax.set_ylabel('Mean LHIPA')
ax.set_title('Clicked vs Non-Clicked')
ax.legend(fontsize=9)
ax.set_xticks(range(10))

# (c) Sample size by position
ax = axes[2]
n_per_pos = [sum(1 for r in has_lhipa if r['position'] == p) for p in range(10)]
n_short = [sum(1 for r in result_rows if r['position'] == p and r['result_lhipa'] is None) for p in range(10)]
ax.bar(range(10), n_per_pos, color='#6366f1', alpha=0.7, label=f'≥{min_samples} samples')
ax.bar(range(10), n_short, bottom=n_per_pos, color='#d4d4d8', alpha=0.7, label=f'<{min_samples} samples')
ax.set_xlabel('Result Position')
ax.set_ylabel('Count')
ax.set_title('Sample Size (LHIPA computable?)')
ax.legend(fontsize=8)
ax.set_xticks(range(10))

plt.tight_layout()
plt.savefig('plot_lhipa4_per_result.png', dpi=200, bbox_inches='tight')
plt.show()

# Stats
c_lhipa = [r['result_lhipa'] for r in has_lhipa if r['is_clicked'] == 1]
nc_lhipa = [r['result_lhipa'] for r in has_lhipa if r['is_clicked'] == 0]
if c_lhipa and nc_lhipa:
    U, p_cn = stats.mannwhitneyu(c_lhipa, nc_lhipa, alternative='two-sided')
    print(f'\nClicked LHIPA: {np.mean(c_lhipa):.6f} (N={len(c_lhipa)})')
    print(f'Non-clicked LHIPA: {np.mean(nc_lhipa):.6f} (N={len(nc_lhipa)})')
    print(f'Mann-Whitney: p={p_cn:.4e}')
    print(f'Direction: {"Clicked=LOWER LHIPA (MORE load)" if np.mean(c_lhipa) < np.mean(nc_lhipa) else "Clicked=HIGHER LHIPA (LESS load)"}')

## Summary

### LHIPA validates against all four behavioral proxies

| Behavioral measure | Spearman rho | p | Direction |
|---|---|---|---|
| Trial duration | -0.650 | ~0 | Longer trials = lower LHIPA = more load |
| Fixation count | -0.621 | ~0 | More fixations = more load |
| Regression count | -0.435 | 3.5e-126 | More regressions = more load |
| Click position | -0.088 | 4.1e-6 | Deeper click = more load |

Trial-level correlation: **ρ = -0.088 (p = 4.1e-6, N ≈ 2,700)**. LHIPA is flat across positions 0–8 (delta = 0.0008) with a step down at boundary positions 9–10. The position-mean correlation (ρ = -0.903, p = 0.0003) is computed on N=10 aggregated means and is driven entirely by the boundary step, not a monotonic decline.

### Per-result LHIPA — use with caution

78% of result-level segments (13,649 / 17,393) have sufficient samples (>=64) for wavelet decomposition, but Duchowski (2026) established that stable LF/HF frequency band separation requires 7.5–10 seconds minimum. Per-result segments (~2s) are well below this threshold. **The per-result LHIPA values in this notebook should not be used for positional claims.**

For per-position cognitive load, use the Butterworth IIR method in **notebook 14**, which has a validated 1-second minimum window. That analysis reveals cognitive load *decreases* with position (ρ = −0.618) — the opposite of what unstable per-result LHIPA suggested. Three methods converge on this direction: Butterworth LF/HF, DWT LF/HF at proper window sizes, and raw mean pupil diameter.

### Comparison with Shi et al. (CHIIR 2025)

Shi, Jayawardena & Gwizdka applied LHIPA to document relevance assessment at 300 Hz (Tobii TX-300) with experimentally controlled relevance levels. We apply the same algorithm to naturalistic SERP browsing at 150 Hz (Gazepoint GP3 HD). Their data is not publicly available; their task used pre-labeled CLEF eHealth documents with post-task perceived relevance ratings. AdSERP lacks both, so their specific mismatch finding (topical x perceived relevance) cannot be directly replicated. What we contribute is: (1) LHIPA validation on a public dataset, (2) demonstration at 150 Hz, (3) the click-position × LHIPA relationship at trial level, clarifying that the aggregate trend is driven by boundary positions 9–10.

### For AdSERP users

The LHIPA implementation in this notebook and in `scripts/compute_lhipa.py` can be applied directly to the AdSERP pupil data (`pupil-data/*.csv`). Key preprocessing notes:

1. **Blink removal**: Use LPV/RPV validity flags with a 200ms exclusion window. Interpolate gaps.
2. **Eye combination**: Mean of valid eyes per sample; single eye if only one is valid.
3. **BPOGY clamp**: Gazepoint FPOGY/BPOGY values can exceed screen height (24.5% of samples). Clamp to `[0, screen_height]` before mapping to page-space positions.
4. **Minimum segment length**: 64 samples for wavelet decomposition, but 7.5s (1,125 samples at 150 Hz) for reliable LF/HF separation (Duchowski 2026). For shorter segments, use Butterworth IIR (`scripts/compute_butterworth_lfhf.py`).
5. **Wavelet**: sym16 with periodization mode, per Duchowski et al. (2020).